In [0]:
from pyspark.sql import functions as F
from delta.tables import DeltaTable

In [0]:
%run /Workspace/consolidated_pipeline/consolidate_pipeline/1_setup/utilities

In [0]:
print(bronze_schema,silver_schema,gold_schema)

In [0]:
dbutils.widgets.text("catalog","fmcg","Catalog")
dbutils.widgets.text("data_source","customer","Data Source")

In [0]:
catalog=dbutils.widgets.get('catalog')
data_source=dbutils.widgets.get('data_source')

print(catalog,data_source)

base_path=f"s3://sportsbar-pro/{data_source}/*.csv"
print(base_path)

In [0]:
df=(
    spark.read.format('csv')
    .option('header',True)
    .option('inferschema',True)
    .load(base_path)
    .withColumn('read_timestamp',F.current_timestamp())
    .select('*','_metadata.file_name','_metadata.file_size')
)

df.show(truncate=False)

In [0]:
print(f"{catalog}.{bronze_schema}.{data_source}")
(
    df.write
    .format('delta')
    .option('delta.enableChangeDataFeed','true')
    .mode('overwrite')
    .saveAsTable(f"{catalog}.{bronze_schema}.{data_source}")
)



### Silver Processing

In [0]:
df_bronze=spark.sql(f"select * from {catalog}.{bronze_schema}.{data_source}")

df_bronze.show(10)

In [0]:
df_duplicates=df_bronze.groupBy('customer_id').count().filter(F.col('count')>1)

df_duplicates.show()

In [0]:
print("Row before Removing duplicates: ",df_bronze.count())

df_silver=df_bronze.dropDuplicates(['customer_id'])
print('Rows after removeing duplicate: ',df_silver.count())

In [0]:
df_silver.filter(F.col('customer_name')!=F.trim(F.col('customer_name'))).show()

In [0]:
df_silver=df_silver.withColumn(
    'customer_name',
    F.trim(F.col('customer_name'))
)

In [0]:
df_silver.filter(F.col('customer_name')!=F.trim(F.col('customer_name'))).show()

In [0]:
df_silver.select('city').distinct().show()

df_silver.show()

In [0]:
city_mapping={
    "Bengaluruu":"Bengaluru",
    "Bengalore":"Bengaluru",

    'Hyderabadd':'Hyderabad',
    'Hyderbad':'Hyderabad',

    'NewDelhi':'New Delhi',
    'NewDheli':'New Delhi',
    'NewDelhee':'New Delhi'
}

allowed=['Bengaluru','Hyderabad','New Delhi']

df_silver=(
    df_silver
    .replace(city_mapping,subset=['city'])
    .withColumn(
        "city",
        F.when(F.col("city").isNull(),None)
        .when(F.col("city").isin(allowed),F.col("city"))
        .otherwise(None)
    )
    
)
df_silver.show()

In [0]:
df_silver.select('city').distinct().show()

In [0]:
df_silver.select('customer_name').distinct().show()

In [0]:
df_silver=df_silver.withColumn(
    'customer_name',
    F.when(F.col('customer_name').isNull(),None)
    .otherwise(F.initcap('customer_name'
    ))
)

df_silver.select('customer_name').distinct().show()

In [0]:
df_silver.filter(F.col('city').isNull()).show(truncate=False)

In [0]:
null_customer_names=['Sprintx Nutrition','Zenathlete Foods','Primefuel Nutrition','Recovery Lane']

df_silver.filter(F.col('customer_name').isin(null_customer_names)).show(truncate=False)

In [0]:

# Business Confirmation Note: City corrections confirmed by business team
customer_city_fix = {
    # Sprintx Nutrition
    789403: "New Delhi",

    # Zenathlete Foods
    789420: "Bengaluru",

    # Primefuel Nutrition
    789521: "Hyderabad",

    # Recovery Lane
    789603: "Hyderabad"
}


df_fix=spark.createDataFrame(
    [(k,v) for k,v in customer_city_fix.items()],
    ['customer_id','fixed_city']
)
df_fix.show()

In [0]:
df_silver=(
    df_silver.join(df_fix,'customer_id','left')
    .withColumn(
        'city',
        F.coalesce('city','fixed_city')
    )
    .drop('fixed_city')
)

df_silver.show(50)

In [0]:
df_silver=df_silver.withColumn('customer_id',F.col('customer_id').cast('string'))
print(df_silver.printSchema())

In [0]:
df_silver=(
    df_silver.withColumn(
        'customer',
        F.concat_ws('-','customer_name',F.coalesce(F.col('city'),F.lit('Unkown')))
    )
    .withColumn('market',F.lit('India'))
    .withColumn('platform',F.lit('Sports Bar'))
    .withColumn('channel',F.lit('Acquistion'))
)
df_silver.show(truncate=False)

In [0]:
(df_silver.write
.format('delta')
.option('delta.enableChangeDataFeed','true')
.option('mergeSchema','true')
.mode('overwrite')
.saveAsTable(f'{catalog}.{silver_schema}.{data_source}')
)

### Gold Proceessing

In [0]:
df_silver=spark.sql(f"SELECT * FROM {catalog}.{silver_schema}.{data_source};")

df_gold=df_silver.select('customer_id','customer_name','city','customer','market','platform','channel')

df_gold.show()

In [0]:
(
    df_gold.write
    .format('delta')
    .option('delta.enableChangeDataFeed','true')
    .mode('overwrite')
    .saveAsTable(f'{catalog}.{gold_schema}.sb_dim_{data_source}')
)

In [0]:
delta_table= DeltaTable.forName(spark,'fmcg.gold.dim_customers')
df_child_customers=spark.table('fmcg.gold.sb_dim_customers').select(
    F.col('customer_id').alias('customer_code'),
    'customer',
    'market',
    'platform',
    'channel'
)

In [0]:
delta_table.alias('target').merge(
    source=df_child_customers.alias('source'),
    condition='target.customer_code =source.customer_code'
).whenMatchedUpdateAll().whenNotMatchedInsertAll().execute()